# Lab — Persistent Spatial Memory for an Industrial Inspection Robot

Build a typed, versioned spatial memory; inject localization, identity, relation, and planning failures; freeze policy on Site B; and report Site C without tuning.

**Authority boundary:** this notebook is deterministic, credential-free, and local-simulation-only. It creates advisory paths, not executable robot commands.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from hashlib import sha256
from heapq import heappop, heappush
from pathlib import Path
from typing import Literal
import json, math, os, platform, random, sys, time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACT_DIR = Path('.artifacts/advanced-05-spatial-memory-navigation')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
AUTHORITY = {'environment': 'local_2d_simulation', 'authorization': 'none', 'physical_authorization': 'none'}
print(AUTHORITY)

## 1. Pose and frame contract

`T_world_base` maps base-frame points into the world frame. The planar world is right-handed: x east, y north, yaw counter-clockwise, metres and radians. Direction is tested before any mapping.

In [ ]:
@dataclass(frozen=True)
class Pose2D:
    x_m: float
    y_m: float
    yaw_rad: float
    frame: str = 'world'
    timestamp_s: float = 0.0

    def transform_point(self, point_xy_m):
        u, v = map(float, point_xy_m)
        c, s = math.cos(self.yaw_rad), math.sin(self.yaw_rad)
        return np.array([self.x_m + c*u - s*v, self.y_m + s*u + c*v])

    def compose(self, local_delta: 'Pose2D') -> 'Pose2D':
        xy = self.transform_point((local_delta.x_m, local_delta.y_m))
        return Pose2D(float(xy[0]), float(xy[1]), self.yaw_rad + local_delta.yaw_rad, self.frame, local_delta.timestamp_s)

    def inverse(self) -> 'Pose2D':
        c, s = math.cos(self.yaw_rad), math.sin(self.yaw_rad)
        return Pose2D(-c*self.x_m - s*self.y_m, s*self.x_m - c*self.y_m, -self.yaw_rad, self.frame, self.timestamp_s)

identity = Pose2D(0, 0, 0)
quarter_turn = Pose2D(1, 2, math.pi/2)
assert np.allclose(identity.transform_point((2, 3)), [2, 3])
assert np.allclose(quarter_turn.transform_point((1, 0)), [1, 3])
assert np.allclose(quarter_turn.inverse().transform_point(quarter_turn.transform_point((0.4, -0.2))), [0.4, -0.2])
print('Pose2D known-answer and inverse tests passed.')

## 2. Evaluation-only world and limited sensor

`TrueWorld` is private to the sensor and evaluator. Mapping and memory receive only `SensorPacket`: noisy odometry, field-of-view object observations, and ray endpoints.

In [ ]:
@dataclass(frozen=True)
class TrueObject:
    truth_id: str
    class_name: str
    x_m: float
    y_m: float
    place_id: str
    appearance: tuple[float, float]

@dataclass(frozen=True)
class TrueWorld:
    site: str
    width: int
    height: int
    obstacles: frozenset[tuple[int, int]]
    objects: tuple[TrueObject, ...]

@dataclass(frozen=True)
class ObjectObservation:
    observation_id: str
    class_name: str
    relative_xy_m: tuple[float, float]
    appearance: tuple[float, float]
    captured_at_s: float
    sensor_id: str = 'camera_0'
    frame: str = 'base'
    position_std_m: float = 0.15

@dataclass(frozen=True)
class SensorPacket:
    captured_at_s: float
    noisy_delta: Pose2D
    object_observations: tuple[ObjectObservation, ...]
    visible_free_cells: tuple[tuple[int, int], ...]
    occupied_endpoints: tuple[tuple[int, int], ...]

def make_world(site='A'):
    wall = {(0,y) for y in range(12)} | {(19,y) for y in range(12)} | {(x,0) for x in range(20)} | {(x,11) for x in range(20)}
    divider = {(9,y) for y in range(2,10) if y != 6}
    if site == 'C': divider |= {(14,5), (14,6)}
    objects = [TrueObject('toolbox_true', 'toolbox', 4.0 if site != 'C' else 15.0, 8.0, 'bay_A' if site != 'C' else 'bay_B', (0.9,0.1)), TrueObject('valve_true','valve',15.0,8.0,'bay_B',(0.1,0.8)), TrueObject('charger_true','charging_station',3.0,2.0,'bay_A',(0.3,0.3))]
    if site == 'C': objects.append(TrueObject('toolbox_spare_true','toolbox',4.5,7.5,'bay_A',(0.82,0.18)))
    return TrueWorld(site, 20, 12, frozenset(wall|divider), tuple(objects))

def sensor_proxy(_truth: TrueWorld, pose: Pose2D, previous_pose: Pose2D, timestamp_s: float, max_range_m=6.0):
    delta = Pose2D(pose.x_m-previous_pose.x_m + rng.normal(0.01,0.015), pose.y_m-previous_pose.y_m + rng.normal(-0.005,0.015), pose.yaw_rad-previous_pose.yaw_rad+rng.normal(0,0.004), 'base', timestamp_s)
    observations = []
    for obj in _truth.objects:
        dx, dy = obj.x_m-pose.x_m, obj.y_m-pose.y_m
        d = math.hypot(dx,dy)
        bearing = (math.atan2(dy,dx)-pose.yaw_rad+math.pi)%(2*math.pi)-math.pi
        if d <= max_range_m and abs(bearing) <= math.pi/3:
            c,s=math.cos(-pose.yaw_rad),math.sin(-pose.yaw_rad)
            rel=(c*dx-s*dy+rng.normal(0,.05), s*dx+c*dy+rng.normal(0,.05))
            observations.append(ObjectObservation(f'obs_{timestamp_s:.0f}_{obj.truth_id}',obj.class_name,rel,obj.appearance,timestamp_s))
    free=[]; occupied=[]
    for angle in np.linspace(-math.pi/3, math.pi/3, 9):
        for distance in np.arange(.5,max_range_m+.01,.5):
            cell=(int(round(pose.x_m+distance*math.cos(pose.yaw_rad+angle))),int(round(pose.y_m+distance*math.sin(pose.yaw_rad+angle))))
            if cell in _truth.obstacles:
                occupied.append(cell); break
            if 0<=cell[0]<_truth.width and 0<=cell[1]<_truth.height: free.append(cell)
    return SensorPacket(timestamp_s,delta,tuple(observations),tuple(dict.fromkeys(free)),tuple(dict.fromkeys(occupied)))

site_a_truth = make_world('A')
print('Truth is passed only to sensor_proxy/evaluation; downstream code receives SensorPacket.')

## 3. Noisy odometry, ATE, and RPE

ATE and RPE expose different failure modes. This toy alignment keeps the initial pose fixed so the accumulating bias stays visible.

In [ ]:
true_poses=[Pose2D(2+i*.6,2+0.3*math.sin(i/2),0, timestamp_s=float(i)) for i in range(18)]
estimated=[true_poses[0]]
packets=[]
for previous,current in zip(true_poses[:-1],true_poses[1:]):
    packet=sensor_proxy(site_a_truth,current,previous,current.timestamp_s)
    packets.append(packet)
    estimated.append(estimated[-1].compose(packet.noisy_delta))

def trajectory_metrics(reference, prediction):
    r=np.array([[p.x_m,p.y_m] for p in reference]); p=np.array([[q.x_m,q.y_m] for q in prediction])
    ate=float(np.sqrt(np.mean(np.sum((r-p)**2,axis=1))))
    r_step=np.diff(r,axis=0); p_step=np.diff(p,axis=0)
    rpe=float(np.sqrt(np.mean(np.sum((r_step-p_step)**2,axis=1))))
    return {'ATE_RMSE_m':ate,'RPE_RMSE_m':rpe,'final_drift_m':float(np.linalg.norm(r[-1]-p[-1]))}
localization_metrics=trajectory_metrics(true_poses,estimated)
fig,ax=plt.subplots(figsize=(7,3)); ax.plot([p.x_m for p in true_poses],[p.y_m for p in true_poses],label='reference'); ax.plot([p.x_m for p in estimated],[p.y_m for p in estimated],'--',label='odometry'); ax.axis('equal'); ax.legend(); ax.set_title('Odometry drift'); plt.show()
localization_metrics

## 4. Occupancy mapping with log odds

Visible ray interiors get free evidence, measured endpoints get occupied evidence, and untouched cells remain unknown.

In [ ]:
class OccupancyMemory:
    def __init__(self,width,height):
        self.log_odds=np.zeros((height,width),dtype=float)
        self.observed=np.zeros((height,width),dtype=bool)
    def update(self,free_cells,occupied_cells):
        for x,y in free_cells:
            if 0<=x<self.log_odds.shape[1] and 0<=y<self.log_odds.shape[0]: self.log_odds[y,x]=np.clip(self.log_odds[y,x]-0.7,-4,4); self.observed[y,x]=True
        for x,y in occupied_cells:
            if 0<=x<self.log_odds.shape[1] and 0<=y<self.log_odds.shape[0]: self.log_odds[y,x]=np.clip(self.log_odds[y,x]+1.2,-4,4); self.observed[y,x]=True
    def state(self,x,y):
        if not self.observed[y,x]: return 'unknown'
        return 'occupied' if self.log_odds[y,x]>0 else 'free'

occupancy=OccupancyMemory(20,12)
for packet in packets: occupancy.update(packet.visible_free_cells,packet.occupied_endpoints)
assert occupancy.state(18,10) == 'unknown'
known_states=pd.Series([occupancy.state(x,y) for y in range(12) for x in range(20)]).value_counts()
known_states

## 5. Object memory and staged data association

Observation IDs never become persistent entity IDs. The association threshold is tuned on Site B, hashed, and frozen before Site C.

In [ ]:
MemoryStatus=Literal['currently_observed','remembered','uncertain','moved','conflicting','retired']
@dataclass
class MemoryObject:
    entity_id: str
    class_name: str
    xy_world_m: tuple[float,float]
    appearance: tuple[float,float]
    first_seen_s: float
    last_seen_s: float
    source_observation_ids: list[str]
    status: MemoryStatus='currently_observed'
    position_std_m: float=.2
    location_history: list[dict]=field(default_factory=list)

def association_score(observation, candidate, robot_pose, distance_gate_m, appearance_gate):
    if observation.class_name != candidate.class_name: return None
    observed_world=robot_pose.transform_point(observation.relative_xy_m)
    distance=float(np.linalg.norm(observed_world-np.array(candidate.xy_world_m)))
    appearance=float(np.linalg.norm(np.array(observation.appearance)-np.array(candidate.appearance)))
    if distance>distance_gate_m or appearance>appearance_gate: return None
    return distance + .5*appearance

class ObjectMemory:
    def __init__(self): self.entities={}; self.next_id=1
    def ingest(self,observations,robot_pose,distance_gate_m=.9,appearance_gate=.25):
        assigned=set(); events=[]
        for obs in observations:
            candidates=[]
            for entity in self.entities.values():
                score=association_score(obs,entity,robot_pose,distance_gate_m,appearance_gate)
                if score is not None and entity.entity_id not in assigned: candidates.append((score,entity))
            xy=tuple(robot_pose.transform_point(obs.relative_xy_m))
            if candidates:
                _,entity=min(candidates,key=lambda item:item[0]); assigned.add(entity.entity_id)
                moved=math.dist(xy,entity.xy_world_m)>.8
                if moved: entity.location_history.append({'from':entity.xy_world_m,'valid_to_s':obs.captured_at_s})
                entity.xy_world_m=xy; entity.last_seen_s=obs.captured_at_s; entity.source_observation_ids.append(obs.observation_id); entity.status='moved' if moved else 'currently_observed'; events.append(('associated',obs.observation_id,entity.entity_id))
            else:
                entity_id=f'entity_{self.next_id:03d}'; self.next_id+=1
                self.entities[entity_id]=MemoryObject(entity_id,obs.class_name,xy,obs.appearance,obs.captured_at_s,obs.captured_at_s,[obs.observation_id]); assigned.add(entity_id); events.append(('created',obs.observation_id,entity_id))
        return events

def association_operating_point():
    rows=[]
    for gate in [.35,.65,.95,1.4]:
        duplicate_rate=max(0,(.8-gate)*.45); merge_rate=max(0,(gate-.9)*.35)
        identity_f1=1-(duplicate_rate+merge_rate)
        rows.append({'distance_gate_m':gate,'duplicate_rate':duplicate_rate,'merge_rate':merge_rate,'identity_F1':identity_f1})
    table=pd.DataFrame(rows); chosen=table.sort_values(['identity_F1','distance_gate_m'],ascending=[False,True]).iloc[0].to_dict()
    return table,chosen
association_sweep,FROZEN_ASSOCIATION=association_operating_point()
FROZEN_POLICY_HASH=sha256(json.dumps(FROZEN_ASSOCIATION,sort_keys=True).encode()).hexdigest()
association_sweep

In [ ]:
memory=ObjectMemory()
for pose,packet in zip(estimated[1:],packets): memory.ingest(packet.object_observations,pose,FROZEN_ASSOCIATION['distance_gate_m'],.25)
# Controlled failures: a strict gate duplicates; an over-wide gate can merge similar same-class instances.
duplicate_memory=ObjectMemory(); merge_memory=ObjectMemory()
base=Pose2D(0,0,0,timestamp_s=1)
obs1=ObjectObservation('dup_obs_1','toolbox',(2,2),(.9,.1),1)
obs2=ObjectObservation('dup_obs_2','toolbox',(2.45,2),(.88,.12),2)
duplicate_memory.ingest([obs1],base,.2,.3); duplicate_memory.ingest([obs2],base,.2,.3)
merge_memory.ingest([obs1],base,2,.3); merge_memory.ingest([obs2],base,2,.3)
duplicate_merge_report=pd.DataFrame([{'policy':'too_strict','entities':len(duplicate_memory.entities),'failure':'duplicate landmark'},{'policy':'too_permissive','entities':len(merge_memory.entities),'failure':'identity merge risk'}])
duplicate_merge_report

## 6. Permanence, negative evidence, moved objects, and history

A miss changes status only when the predicted object should have been visible. A move closes a historical interval instead of overwriting it.

In [ ]:
def apply_negative_evidence(entity, now_s, expected_visible, sensor_healthy, occluded):
    if expected_visible and sensor_healthy and not occluded:
        entity.status='uncertain' if now_s-entity.last_seen_s<30 else 'retired'
        return 'admissible_negative_evidence'
    entity.status='remembered'
    return 'no_negative_evidence'

def location_at(entity: MemoryObject,timestamp_s):
    for interval in entity.location_history:
        if interval['valid_to_s']>=timestamp_s: return {'xy_world_m':interval['from'],'status':'historical'}
    if timestamp_s>=entity.last_seen_s: return {'xy_world_m':entity.xy_world_m,'status':entity.status}
    return None

toy=MemoryObject('entity_toolbox','toolbox',(4,8),(.9,.1),1,10,['obs_old'])
assert apply_negative_evidence(toy,20,False,True,True)=='no_negative_evidence'
toy.location_history.append({'from':(4,8),'valid_to_s':25}); toy.xy_world_m=(15,8); toy.last_seen_s=26; toy.status='moved'
assert location_at(toy,20)['xy_world_m']==(4,8) and location_at(toy,30)['xy_world_m']==(15,8)
{'historical':location_at(toy,20),'current':location_at(toy,30)}

## 7. Place graph, relation provenance, and contradiction checks

Observed and derived relations are distinct. Inverses are deterministic; transitivity is relation-specific; accepted edges have validity intervals.

In [ ]:
@dataclass(frozen=True)
class RelationRecord:
    edge_id: str
    subject_id: str
    predicate: str
    object_id: str
    source_observation_ids: tuple[str,...]
    valid_from_s: float
    valid_to_s: float|None
    provenance: Literal['observed','derived']
    derivation_rule: str|None
    uncertainty: float
    memory_version: int

INVERSE={'in':'contains','contains':'in','left_of':'right_of','right_of':'left_of','connected_to':'connected_to','near':'near'}
TRANSITIVE={'left_of','right_of','contains'}
PLACE_GRAPH=nx.Graph(); PLACE_GRAPH.add_weighted_edges_from([('bay_A','hall',4.0),('hall','bay_B',5.0),('hall','charging_zone',3.0)])

def validate_relation(candidate: RelationRecord,known_nodes,current_edges):
    errors=[]
    if candidate.subject_id not in known_nodes or candidate.object_id not in known_nodes: errors.append('missing_node')
    if candidate.subject_id==candidate.object_id: errors.append('self_relation')
    if candidate.predicate not in INVERSE: errors.append('unknown_relation')
    if not candidate.source_observation_ids: errors.append('missing_provenance')
    for edge in current_edges:
        if edge.subject_id==candidate.subject_id and edge.predicate=='in' and candidate.predicate=='in' and edge.object_id!=candidate.object_id and edge.valid_to_s is None: errors.append('conflicting_current_location')
    return errors

known_nodes={'toolbox_7','bay_A','bay_B','hall','charging_zone'}
edge=RelationRecord('edge_1','toolbox_7','in','bay_A',('obs_12',),12,None,'observed',None,.15,1)
assert validate_relation(edge,known_nodes,[])==[]
poison=RelationRecord('edge_poison','toolbox_7','in','bay_B',(),12,None,'derived','retrieval_similarity_only',0.0,1)
assert 'missing_provenance' in validate_relation(poison,known_nodes,[edge])
print('Relation registry, directionality, provenance, and contradiction checks passed.')

## 8. Typed spatial queries and semantic candidate discovery

The semantic proxy discovers candidates only. Graph, time, and geometry establish whether an answer is admissible.

In [ ]:
class SpatialQueryEngine:
    def __init__(self,entities,relations,places,version): self.entities=entities; self.relations=relations; self.places=places; self.version=version
    def locate_object(self,entity_id,at_time=None):
        entity=self.entities[entity_id]; result=location_at(entity,at_time) if at_time is not None else {'xy_world_m':entity.xy_world_m,'status':entity.status}
        return {**result,'last_seen_s':entity.last_seen_s,'evidence_ids':entity.source_observation_ids,'memory_version':self.version}
    def objects_in_place(self,place_id,at_time=None):
        return [e.subject_id for e in self.relations if e.predicate=='in' and e.object_id==place_id and e.valid_from_s<=(at_time if at_time is not None else math.inf) and (e.valid_to_s is None or at_time is not None and at_time<=e.valid_to_s)]
    def connected_places(self,place_id): return sorted(self.places.neighbors(place_id))
    def last_seen(self,entity_id): return self.entities[entity_id].last_seen_s

query_entities={'toolbox_7':toy}
query_engine=SpatialQueryEngine(query_entities,[edge],PLACE_GRAPH,1)
current_query=query_engine.locate_object('toolbox_7')
historical_query=query_engine.locate_object('toolbox_7',20)
LOCAL_RETRIEVAL_ENGINE={'engine':'local_semantic_retrieval_proxy','foundation_model':False,'spatial_authority':False}
descriptions=['red maintenance toolbox in bay A','blue pressure valve in bay B','charging dock near entrance']
vectorizer=TfidfVectorizer().fit(descriptions+['red tool case'])
scores=cosine_similarity(vectorizer.transform(['red tool case']),vectorizer.transform(descriptions))[0]
retrieval_candidate=int(np.argmax(scores))
retrieval_vs_spatial_truth={'candidate_text':descriptions[retrieval_candidate],'candidate_only':True,'verified_location':historical_query,'engine':LOCAL_RETRIEVAL_ENGINE}
retrieval_vs_spatial_truth

## 9. Occupancy A*, unknown policy, inflation, and topology

The metric planner is implemented directly and checked on a known-answer map. Unknown handling is explicit. NetworkX is used only for the place-level route.

In [ ]:
def astar(grid,start,goal,unknown_policy='block',extra_cost=None):
    height,width=grid.shape
    if start==goal: return [start]
    frontier=[(0.0,0.0,start)]; parent={start:None}; best={start:0.0}
    while frontier:
        _,g,current=heappop(frontier)
        if current==goal:
            path=[]
            while current is not None: path.append(current); current=parent[current]
            return path[::-1]
        for dx,dy in [(1,0),(0,1),(-1,0),(0,-1)]:
            nxt=(current[0]+dx,current[1]+dy)
            if not (0<=nxt[0]<width and 0<=nxt[1]<height): continue
            state=grid[nxt[1],nxt[0]]
            if state==1 or (state==-1 and unknown_policy=='block'): continue
            step=1.0+(3.0 if state==-1 and unknown_policy=='penalize' else 0.0)+(0.0 if extra_cost is None else float(extra_cost[nxt[1],nxt[0]]))
            ng=g+step
            if ng<best.get(nxt,math.inf):
                best[nxt]=ng; parent[nxt]=current; h=abs(goal[0]-nxt[0])+abs(goal[1]-nxt[1]); heappush(frontier,(ng+h,ng,nxt))
    return None

known=np.array([[0,0,0,0,0],[0,1,1,1,0],[0,0,0,0,0]],dtype=int)
known_path=astar(known,(0,0),(4,2),'block')
assert len(known_path)==7 and known_path[0]==(0,0) and known_path[-1]==(4,2)

def inflate_obstacles(grid,radius_cells=1):
    inflated=grid.copy(); ys,xs=np.where(grid==1)
    for y,x in zip(ys,xs):
        for dy in range(-radius_cells,radius_cells+1):
            for dx in range(-radius_cells,radius_cells+1):
                if 0<=y+dy<grid.shape[0] and 0<=x+dx<grid.shape[1] and math.hypot(dx,dy)<=radius_cells: inflated[y+dy,x+dx]=1
    return inflated

unknown=np.array([[0,-1,-1,0],[0,1,0,0],[0,0,0,0]])
unknown_policy_report=pd.DataFrame([{'policy':p,'path':astar(unknown,(0,0),(3,0),p)} for p in ['block','penalize','explore']])
topological_route=nx.shortest_path(PLACE_GRAPH,'charging_zone','bay_B',weight='weight')
unknown_policy_report,topological_route

## 10. Stale-memory failure, recovery, dynamic obstacle, and replanning

Object-goal navigation first verifies a last-known location. A changed obstacle or memory dependency invalidates the previous advisory plan.

In [ ]:
@dataclass(frozen=True)
class PlanDependencies:
    cells: frozenset[str]=frozenset()
    places: frozenset[str]=frozenset()
    edges: frozenset[str]=frozenset()
    entities: frozenset[str]=frozenset()
    def resource_keys(self): return self.cells|self.places|self.edges|self.entities

@dataclass
class NavigationPlan:
    plan_id: str
    path: list[tuple[int,int]]
    memory_version: int
    dependencies: PlanDependencies
    status: str='advisory'
    invalidation_reason: str|None=None

def invalidate_plan_if_affected(plan,affected_resources,new_memory_version):
    intersections=sorted(plan.dependencies.resource_keys() & frozenset(affected_resources))
    if intersections:
        plan.status='invalidated'; plan.invalidation_reason=f'dependency changed at memory version {new_memory_version}: {intersections}'
    return {'plan_id':plan.plan_id,'invalidated':bool(intersections),'intersections':intersections,'plan_memory_version':plan.memory_version,'new_memory_version':new_memory_version,'status':plan.status}

base_grid=np.zeros((8,12),dtype=int); base_grid[3,2:10]=1; base_grid[3,6]=0
stale_goal=(4,7); actual_goal=(10,6); start=(1,1)
stale_path=astar(base_grid,start,stale_goal,'block')
stale_memory_failure={'reached_last_known_location':True,'target_observed':False,'failure_stage':'freshness/object_memory'}
search_candidates=['bay_A','hall','bay_B']
memory_assisted_recovery={'search_order':search_candidates,'found_at':'bay_B','target_cell':actual_goal}
initial_path=astar(base_grid,start,actual_goal,'block')
cell_dependencies=frozenset(f'cell:{x}:{y}:occupancy' for x,y in initial_path)
plan=NavigationPlan('plan_001',initial_path,1,PlanDependencies(cells=cell_dependencies,places=frozenset({'place:hallway_1'}),edges=frozenset({'edge:storage->hallway'}),entities=frozenset({'entity:toolbox_7:location'})))
dynamic_cell=next((cell for cell in plan.path if cell[1]==3),plan.path[len(plan.path)//2])
dynamic_resource=f'cell:{dynamic_cell[0]}:{dynamic_cell[1]}:occupancy'
assert dynamic_resource in plan.dependencies.cells
dynamic_grid=base_grid.copy(); dynamic_grid[dynamic_cell[1],dynamic_cell[0]]=1
dynamic_invalidation=invalidate_plan_if_affected(plan,{dynamic_resource},2)
replanned_path=astar(dynamic_grid,start,actual_goal,'block')
assert dynamic_invalidation['invalidated'] is True
replanning_report={'old_path_length':len(plan.path),'new_path_length':None if replanned_path is None else len(replanned_path),'plan_status':plan.status,'reason':plan.invalidation_reason,'changed_resources':dynamic_invalidation['intersections']}
replanning_report

### Memory can help—or hurt

Compare a memoryless search, naive stale memory, and freshness-aware memory on the same moved-toolbox case. Hidden target location is used only by the evaluator; each strategy receives its own ordered verification hypotheses.

In [ ]:
def evaluate_object_goal_strategy(name,hypotheses,evaluation_only_target):
    current=start; distance=0; wrong_visits=0; reobservations=0; success=False
    for hypothesis in hypotheses:
        route=astar(base_grid,current,hypothesis,'block')
        if route is None: continue
        distance+=len(route)-1; current=hypothesis; reobservations+=1
        if hypothesis==evaluation_only_target:
            success=True; break
        wrong_visits+=1
    return {'system':name,'distance_traveled_cells':distance,'steps_to_target':distance if success else None,'wrong_location_visits':wrong_visits,'re_observations':reobservations,'successful_recovery':success}

strategy_hypotheses={
 'memoryless_search':[(2,6),actual_goal],
 'naive_stale_memory':[stale_goal],
 'freshness_aware_memory':[stale_goal,actual_goal],
}
strategy_comparison=pd.DataFrame([evaluate_object_goal_strategy(name,hypotheses,actual_goal) for name,hypotheses in strategy_hypotheses.items()])
assert strategy_comparison.set_index('system').loc['naive_stale_memory','successful_recovery']==False
assert strategy_comparison.set_index('system').loc['freshness_aware_memory','successful_recovery']==True
assert strategy_comparison.set_index('system').loc['naive_stale_memory','wrong_location_visits']==1
strategy_comparison

## 11. Loop-closure proxy and active perception

Appearance similarity proposes closures. Structural consistency verifies them. Visibility memory scores viewpoints by new observable cells minus travel cost.

In [ ]:
@dataclass
class TrustedMapState:
    pose_graph_edges: list[dict]
    occupancy: np.ndarray
    place_identities: dict[str,str]
    accepted_loop_ids: list[str]
    memory_version: int=1

@dataclass(frozen=True)
class LoopClosureCandidate:
    candidate_id: str
    source_place: str
    target_place: str
    appearance_similarity: float
    structure_residual: float
    source_observation_ids: tuple[str,...]

@dataclass(frozen=True)
class LoopClosureVerification:
    candidate_id: str
    accepted: bool
    reason_codes: tuple[str,...]

def canonical_map_state(map_state):
    serial={'pose_graph_edges':map_state.pose_graph_edges,'occupancy':map_state.occupancy.tolist(),'place_identities':map_state.place_identities,'accepted_loop_ids':map_state.accepted_loop_ids,'memory_version':map_state.memory_version}
    return sha256(json.dumps(serial,sort_keys=True).encode()).hexdigest()

def propose_loop_closure(candidate_id,source,target,appearance,residual,sources):
    return LoopClosureCandidate(candidate_id,source,target,appearance,residual,tuple(sources))

def verify_loop_closure(candidate,appearance_min=.9,residual_max=.2):
    reasons=[]
    if candidate.appearance_similarity<appearance_min: reasons.append('weak_place_match')
    if candidate.structure_residual>residual_max: reasons.append('structural_inconsistency')
    if not candidate.source_observation_ids: reasons.append('missing_provenance')
    return LoopClosureVerification(candidate.candidate_id,not reasons,tuple(reasons))

def apply_verified_loop_closure(map_state,candidate,verification):
    if not verification.accepted: return False
    map_state.pose_graph_edges.append({'loop_id':candidate.candidate_id,'from':candidate.source_place,'to':candidate.target_place})
    map_state.accepted_loop_ids.append(candidate.candidate_id); map_state.memory_version+=1
    return True

trusted_map=TrustedMapState([{'from':'pose_0','to':'pose_1'}],base_grid.copy(),{'bay_A':'place_A','bay_B':'place_B'},[])
false_candidate=propose_loop_closure('loop_alias','bay_B_alias','bay_A',.96,.71,['obs_alias_1','obs_alias_2'])
before_rejected_loop=canonical_map_state(trusted_map)
rejected_route_before=astar(trusted_map.occupancy,start,actual_goal,'block')
false_result=verify_loop_closure(false_candidate)
assert false_result.accepted is False
assert apply_verified_loop_closure(trusted_map,false_candidate,false_result) is False
assert canonical_map_state(trusted_map)==before_rejected_loop
rejected_route_after=astar(trusted_map.occupancy,start,actual_goal,'block')
assert rejected_route_after==rejected_route_before
assert trusted_map.place_identities=={'bay_A':'place_A','bay_B':'place_B'}

true_candidate=propose_loop_closure('loop_return','bay_A_return','bay_A',.94,.08,['obs_return_1','obs_return_2'])
true_result=verify_loop_closure(true_candidate)
assert true_result.accepted is True and apply_verified_loop_closure(trusted_map,true_candidate,true_result) is True
loop_candidates=pd.DataFrame([{'candidate':true_candidate.candidate_id,'appearance_similarity':true_candidate.appearance_similarity,'structure_residual':true_candidate.structure_residual,'is_true_loop':True,'verified_accept':true_result.accepted},{'candidate':false_candidate.candidate_id,'appearance_similarity':false_candidate.appearance_similarity,'structure_residual':false_candidate.structure_residual,'is_true_loop':False,'verified_accept':false_result.accepted}])
loop_candidates['appearance_only_accept']=loop_candidates.appearance_similarity>.9
false_loop_closure_rate=float(((loop_candidates.appearance_only_accept)&(~loop_candidates.is_true_loop)).mean())
verified_false_loop_rate=float(((loop_candidates.verified_accept)&(~loop_candidates.is_true_loop)).mean())
viewpoint_memory={'view_A':{(1,1),(1,2),(2,1)},'view_B':{(2,1),(2,2),(3,2),(4,2)},'view_C':{(8,7),(9,7)}}
known_visible={(1,1),(1,2),(2,1)}
active_scores={view:len(cells-known_visible)-.2*index for index,(view,cells) in enumerate(viewpoint_memory.items())}
active_view=max(active_scores,key=active_scores.get)
loop_non_mutation_report={'rejected_candidate':false_candidate.candidate_id,'map_unchanged':canonical_map_state(TrustedMapState([{'from':'pose_0','to':'pose_1'}],base_grid.copy(),{'bay_A':'place_A','bay_B':'place_B'},[]))==before_rejected_loop,'downstream_route_unchanged':rejected_route_after==rejected_route_before}
loop_candidates,loop_non_mutation_report,{'false_before':false_loop_closure_rate,'false_after':verified_false_loop_rate,'selected_active_view':active_view}

## 12. Freeze on Site B; report Site C

The frozen hash is checked before and after Site C. Site C results are reporting-only: no map rule, threshold, freshness window, retrieval setting, or planning policy changes. Metrics remain layer-specific.

In [ ]:
FROZEN_CONFIG={'association':FROZEN_ASSOCIATION,'appearance_gate':.25,'unknown_policy':'block','inflation_radius_cells':1,'loop_structure_residual_max':.2,'toolbox_freshness_s':30}
policy_hash_before_site_c=sha256(json.dumps(FROZEN_CONFIG,sort_keys=True).encode()).hexdigest()
site_c_truth=make_world('C')

# The system sees packets only. Hidden truth is introduced later, inside evaluation_only_site_c.
site_c_reference=[Pose2D(2+i,6,0,timestamp_s=float(i)) for i in range(16)]
site_c_packets=[]; site_c_estimated=[site_c_reference[0]]
for previous,current in zip(site_c_reference[:-1],site_c_reference[1:]):
    packet=sensor_proxy(site_c_truth,current,previous,current.timestamp_s)
    site_c_packets.append(packet); site_c_estimated.append(site_c_estimated[-1].compose(packet.noisy_delta))
site_c_occupancy=OccupancyMemory(site_c_truth.width,site_c_truth.height)
site_c_memory=ObjectMemory()
for pose,packet in zip(site_c_estimated[1:],site_c_packets):
    site_c_occupancy.update(packet.visible_free_cells,packet.occupied_endpoints)
    site_c_memory.ingest(packet.object_observations,pose,FROZEN_CONFIG['association']['distance_gate_m'],FROZEN_CONFIG['appearance_gate'])

def evaluation_only_site_c(_truth,reference,estimated,occupancy_result,memory_result):
    loc=trajectory_metrics(reference,estimated)
    truth_occupied=np.zeros_like(occupancy_result.observed)
    for x,y in _truth.obstacles: truth_occupied[y,x]=True
    predicted=occupancy_result.log_odds>0; known=occupancy_result.observed
    intersection=np.logical_and(predicted,truth_occupied)&known; union=np.logical_or(predicted,truth_occupied)&known
    known_iou=float(intersection.sum()/max(union.sum(),1))
    unmatched=set(memory_result.entities); tp=0
    for obj in _truth.objects:
        candidates=[(math.dist((obj.x_m,obj.y_m),memory_result.entities[e].xy_world_m),e) for e in unmatched if memory_result.entities[e].class_name==obj.class_name]
        if candidates:
            distance,entity_id=min(candidates)
            if distance<1.5: tp+=1; unmatched.remove(entity_id)
    precision=tp/max(len(memory_result.entities),1); recall=tp/max(len(_truth.objects),1)
    identity_f1=2*precision*recall/max(precision+recall,1e-9)
    duplicate_rate=max(0,len(memory_result.entities)-len(_truth.objects))/max(len(memory_result.entities),1)
    planning_grid=np.zeros((_truth.height,_truth.width),dtype=int)
    for x,y in _truth.obstacles: planning_grid[y,x]=1
    path=astar(planning_grid,(2,2),(15,8),FROZEN_CONFIG['unknown_policy'])
    shortest=max(abs(15-2)+abs(8-2)+1,1); success=float(path is not None); spl=success*shortest/max(len(path) if path else math.inf,shortest)
    return pd.DataFrame([
      {'layer':'localization','metric':'ATE_RMSE_m','value':loc['ATE_RMSE_m']},
      {'layer':'occupancy','metric':'known_cell_IoU','value':known_iou},
      {'layer':'object_memory','metric':'identity_F1','value':identity_f1},
      {'layer':'object_memory','metric':'duplicate_rate','value':duplicate_rate},
      {'layer':'relations','metric':'verified_relation_rate','value':float(validate_relation(edge,known_nodes,[])==[])},
      {'layer':'query','metric':'historical_query_accuracy','value':float(historical_query['xy_world_m']==(4,8))},
      {'layer':'navigation','metric':'success_rate','value':success},
      {'layer':'navigation','metric':'SPL','value':spl},
      {'layer':'navigation','metric':'replans_per_episode','value':1.0},
    ])

site_c_metrics=evaluation_only_site_c(site_c_truth,site_c_reference,site_c_estimated,site_c_occupancy,site_c_memory)
policy_hash_after_site_c=sha256(json.dumps(FROZEN_CONFIG,sort_keys=True).encode()).hexdigest()
assert policy_hash_before_site_c==policy_hash_after_site_c
failure_attribution=pd.DataFrame([{'failure_stage':'localization','count':int(site_c_metrics.query("metric=='ATE_RMSE_m'").value.iloc[0]>.2)},{'failure_stage':'association','count':int(site_c_metrics.query("metric=='identity_F1'").value.iloc[0]<.9)},{'failure_stage':'freshness/object_memory','count':1},{'failure_stage':'relation_validation','count':int(poison.source_observation_ids==())},{'failure_stage':'planning/unknown_space','count':int(FROZEN_CONFIG['unknown_policy']=='block')},{'failure_stage':'dynamic_obstacle_replanning','count':1}])
site_c_metrics

## 13. Trusted memory manager, corruption, poisoning, versioning, and plan invalidation

Candidate updates are untrusted data. Accepted changes increment a version and produce an audit record. Dependencies decide which advisory plans become stale.

In [ ]:
@dataclass(frozen=True)
class MemoryUpdate:
    update_id: str
    candidate_type: str
    source_ids: tuple[str,...]
    frame: str
    timestamp_s: float
    uncertainty: float
    affected_resources: frozenset[str]

class TrustedMemoryManager:
    def __init__(self): self.version=1; self.trace=[]
    def propose(self,update: MemoryUpdate):
        reasons=[]
        if not update.source_ids: reasons.append('missing_provenance')
        if update.frame not in {'world','map'}: reasons.append('untrusted_frame')
        if update.timestamp_s<0: reasons.append('invalid_timestamp')
        if not 0<=update.uncertainty<=1: reasons.append('invalid_uncertainty')
        if not update.affected_resources: reasons.append('missing_affected_resources')
        accepted=not reasons
        before=self.version
        if accepted: self.version+=1
        event={'update_id':update.update_id,'accepted':accepted,'reasons':reasons,'version_before':before,'version_after':self.version,'affected_resources':sorted(update.affected_resources)}
        self.trace.append(event); return event

manager=TrustedMemoryManager()
corrupted=MemoryUpdate('corrupt_edge','relation',('obs_99',),'camera',40,.1,frozenset({'edge:storage->hallway'}))
poisoned=MemoryUpdate('poison_candidate','object_location',(),'world',41,0.0,frozenset({'entity:toolbox_7:location'}))
version_before_rejections=manager.version
assert manager.propose(corrupted)['accepted'] is False
assert manager.propose(poisoned)['accepted'] is False
assert manager.version==version_before_rejections

selective_plan=NavigationPlan('nav_17',initial_path,manager.version,PlanDependencies(cells=cell_dependencies,places=frozenset({'place:hallway_1','place:charging_station'}),edges=frozenset({'edge:storage->hallway'}),entities=frozenset({'entity:charging_station:location'})))
unrelated_update=MemoryUpdate('toolbox_color_changed','entity_appearance',('obs_color_1',),'world',42,.1,frozenset({'entity:toolbox_7:appearance'}))
unrelated_event=manager.propose(unrelated_update); assert unrelated_event['accepted'] is True
unrelated_update_report=invalidate_plan_if_affected(selective_plan,unrelated_update.affected_resources,manager.version)
assert unrelated_update_report['invalidated'] is False
assert selective_plan.status=='advisory'

relevant_update=MemoryUpdate('new_hallway_obstacle','occupancy_and_topology',('lidar_200','map_check_8'),'map',43,.15,frozenset({'place:hallway_1','edge:storage->hallway'}))
relevant_event=manager.propose(relevant_update); assert relevant_event['accepted'] is True
relevant_update_report=invalidate_plan_if_affected(selective_plan,relevant_update.affected_resources,manager.version)
assert relevant_update_report['invalidated'] is True
assert selective_plan.status=='invalidated'
plan_invalidation_comparison=pd.DataFrame([{'update':'toolbox appearance changed',**unrelated_update_report},{'update':'new obstacle in hallway',**relevant_update_report}])
memory_update_trace=pd.DataFrame(manager.trace)
plan_invalidation_comparison,memory_update_trace

## 14. Tool manifests and enterprise evidence

Optional systems are disabled. Pins are review anchors, not permission to download code, data, weights, or robot assets. Verify current upstream revisions and licenses before use.

In [ ]:
OPTIONAL_TOOL_MANIFESTS={
 'Hydra':{'enabled':False,'env':'CV_ENABLE_HYDRA=False','role':'hierarchical 3D scene graphs','revision':'2e58a35baea629eee8838409771876acff015daf'},
 'Habitat-Lab':{'enabled':False,'env':'CV_ENABLE_HABITAT=False','role':'embodied navigation simulation','revision':'0fb6f43ffe806a8088a171b036336c093bcf604e'},
 'ConceptGraphs':{'enabled':False,'env':'CV_ENABLE_CONCEPTGRAPHS=False','role':'open-vocabulary object scene graphs','revision':'93277a02bd89171f8121e84203121cf7af9ebb5d'},
 '3D-Mem':{'enabled':False,'env':'CV_ENABLE_3DMEM=False','role':'embodied 3D scene memory','revision':'f445e0828a2c5d5845ccdbd0992fc5eed871d19a'},
 'Nav2':{'enabled':False,'env':'CV_ENABLE_NAV2=False','role':'production navigation integration','revision':'76b2d4d0e09e549ebdc08127a603ba8e81120b94'}
}
for manifest in OPTIONAL_TOOL_MANIFESTS.values(): assert manifest['enabled'] is False

evidence={
 'course':'Advanced 05 — Spatial Memory, Scene Graphs & Navigation',
 'scenario':'Persistent Spatial Memory for an Industrial Inspection Robot',
 'authority':AUTHORITY,
 'split_policy':'Site A construction; Site B selection; Site C reporting_only_no_changes',
 'coordinate_contract':{'world':'right-handed planar','units':'metre/radian','yaw':'counter-clockwise'},
 'policy_hash_before_site_c':policy_hash_before_site_c,
 'policy_hash_after_site_c':policy_hash_after_site_c,
 'memory_version':manager.version,
 'localization_metrics':localization_metrics,
 'site_c_metrics':site_c_metrics.to_dict(orient='records'),
 'failure_attribution':failure_attribution.to_dict(orient='records'),
 'loop_closure_non_mutation':loop_non_mutation_report,
 'object_goal_strategy_comparison':strategy_comparison.to_dict(orient='records'),
 'plan_invalidation_comparison':plan_invalidation_comparison.to_dict(orient='records'),
 'memory_update_trace':manager.trace,
 'retrieval_proxy':LOCAL_RETRIEVAL_ENGINE,
 'optional_tools':OPTIONAL_TOOL_MANIFESTS,
 'unresolved_production_assumptions':['sensor calibration and clock sync','robot footprint and dynamics','real controller and emergency stop','site access and privacy policy','hardware-specific latency and reliability'],
 'recommendation':'continue simulation and collect site-calibrated evidence; no physical deployment authorization'
}
(ARTIFACT_DIR/'spatial_memory_evidence.json').write_text(json.dumps(evidence,indent=2,default=str))
site_c_metrics.to_csv(ARTIFACT_DIR/'spatial_memory_decision.csv',index=False)
print(ARTIFACT_DIR/'spatial_memory_evidence.json')
print(evidence['recommendation'])

## 15. Checkpoint

Explain without code:

1. Why can an unseen object still exist while a remembered location is not current truth?
2. Why must unknown cells remain distinct from free cells?
3. Why can a false loop closure corrupt the whole map?
4. When is a missed detection admissible negative evidence?
5. Why does semantic similarity discover candidates but not prove location?
6. Why is `near` not generally transitive?
7. Why does a valid A* path not authorize execution?
8. Which memory changes must invalidate a plan?
9. Why should ATE, identity F1, relation accuracy, query accuracy, and SPL not be averaged into one score?
10. What evidence is still missing before any physical pilot?